In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib_scalebar.scalebar as scalebar
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import pysal
import esda
import scipy.sparse
import libpysal as lps

from shapely.geometry import LineString, MultiLineString
from scipy.spatial import distance_matrix
from pysal.lib import weights
from pysal.explore import esda
from libpysal.weights import W
from spreg import ML_Lag, ML_Error, GM_Lag
from sklearn.decomposition import PCA
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredDirectionArrows


In [ ]:
# Load spatial data with geopandas
CD = gpd.read_file('data/CommunityDistrict/nycd.shp')
SC = gpd.read_file('data/StreetCenterline/geo_export_5271be70-8d47-4770-8855-7a022145252c.shp')


In [ ]:
# Check CRS of both GeoDataFrames
print("CRS of Community Districts (CD):", CD.crs)
print("CRS of Street Centerline (SC):", SC.crs)

# Align CRS
if CD.crs != SC.crs:
    SC = SC.to_crs(CD.crs)
    print("CRS of SC aligned with CD.")


In [ ]:
# Plotting
fig, ax = plt.subplots(1, 1, figsize=(15, 15))
CD.plot(ax=ax, color='lightblue', edgecolor='black', legend=True, label='Community Districts')
SC.plot(ax=ax, color='grey', linewidth=0.5, legend=True, label='Street Centerlines')

ax.set_title("NYC Community District and Road Network", fontsize=15, pad=20)
north_arrow = AnchoredDirectionArrows(ax.transAxes, 'E', 'N', length=0.15, color='black')
ax.add_artist(north_arrow)

# Adding scale bar for EPSG:2263
# Rough conversion: 1 mile ≈ 5280 feet
scalebar_length = 5280  # 1 mile in feet
scalebar_x = 0.05  # x position in axes coordinates
scalebar_y = 0.05  # y position in axes coordinates
scalebar_text = "1 mile"

ax.legend()
ax.set_axis_off()

# Save
plt.savefig("Layout/NYC_Community_District_and_Road_Network.png", dpi=500)
plt.show()


# Spatial Weighted Matrix

## Step 1: Filter SCs

In [ ]:
# Exclude streets that overlap with CD boundaries
def exclude_overlapping_streets(streets, districts):
    filtered_streets = streets[~streets.geometry.apply(lambda street: any(district.boundary.overlaps(street) for district in districts.geometry))]
    return filtered_streets

SelectedSC = exclude_overlapping_streets(SC, CD)

SelectedSC


In [ ]:
# Exclude streets that not intersecte with CD boundaries
def exclude_non_intersecting_streets(streets, districts):
    filtered_streets = streets[streets.geometry.apply(lambda street: any(district.boundary.intersects(street) for district in districts.geometry))]
    return filtered_streets

SelectedSC = exclude_non_intersecting_streets(SC, CD)

SelectedSC


In [ ]:
# Save
SelectedSC.to_file('data/SelectedStreetCenterline/SelectedSC.shp')


## Step 2: Calculate Weights

### Create a 5-Meter Buffer for Each Street in SelectedSC

In [ ]:
# Create 5 meter buffer for each street
SCBuffer = SelectedSC.copy()
SCBuffer['geometry'] = SCBuffer.geometry.buffer(5)

SCBuffer


### Create a Connection Matrix Based on CDs

In [ ]:
# Create a matrix based on CD
cd_list = CD['BoroCD'].unique()
cd_matrix = pd.DataFrame(index=cd_list, columns=cd_list).fillna(0)
cd_matrix


### Check for Region Overlap and Record Connections and Count All Connections

In [ ]:
for index, street in SCBuffer.iterrows():
    overlapping_cds = CD[CD.geometry.intersects(street.geometry)]['BoroCD'].unique()
    for i in range(len(overlapping_cds)):
        for j in range(i + 1, len(overlapping_cds)):
            cd_matrix.loc[overlapping_cds[i], overlapping_cds[j]] += 1
            cd_matrix.loc[overlapping_cds[j], overlapping_cds[i]] += 1

cd_matrix


### Standardize the Values in the Matrix

In [ ]:
max_value = cd_matrix.max().max()
cd_matrix_normalized = cd_matrix / max_value

cd_matrix_normalized


## Step 3: Visualization

In [ ]:
# Create graph 
G = nx.Graph()

# Add nodes
for cd in cd_list:
    G.add_node(cd)

# Add edges with weights
for i in cd_matrix_normalized.index:
    for j in cd_matrix_normalized.columns:
        if cd_matrix_normalized.loc[i, j] > 0:
            G.add_edge(i, j, weight=cd_matrix_normalized.loc[i, j])

# Visualization
plt.figure(figsize=(15, 15))

# Calculate layout using spring_layout
pos = nx.spring_layout(G)

# Scale positions to increase spacing
pos = {node: (x * 2, y * 2) for (node, (x, y)) in pos.items()}

nx.draw_networkx_nodes(G, pos, node_size=700)
edges = G.edges(data=True)
nx.draw_networkx_edges(G, pos, edgelist=edges, width=[d['weight']*5 for u, v, d in edges])

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=12, font_family="sans-serif")

plt.axis("off")

# Save
save_path = 'Layout'
filename = 'Weighted_Network_Graph_1.png'
if not os.path.exists(save_path):
    os.makedirs(save_path)
plt.savefig(os.path.join(save_path, filename))
plt.show()


In [ ]:
# Function to calculate the product of weights along the shortest path
def path_weight_product(G, source, target):
    try:
        # Get the shortest path
        shortest_path = nx.shortest_path(G, source=source, target=target, weight='weight')
        # Calculate the product of weights along the path
        product = 1
        for i in range(len(shortest_path) - 1):
            edge_weight = G[shortest_path[i]][shortest_path[i + 1]]['weight']
            product *= edge_weight
        return product
    except nx.NetworkXNoPath:
        # Return zero if no path exists
        return 0

# Create a graph from the matrix
G = nx.Graph()

# Add nodes and edges with weights
for i in cd_matrix_normalized.index:
    for j in cd_matrix_normalized.columns:
        if cd_matrix_normalized.loc[i, j] > 0:
            G.add_edge(i, j, weight=cd_matrix_normalized.loc[i, j])

# Update the matrix with new weights for non-adjacent CDs
for i in cd_matrix_normalized.index:
    for j in cd_matrix_normalized.columns:
        if i != j and cd_matrix_normalized.loc[i, j] == 0:
            new_weight = path_weight_product(G, i, j)
            cd_matrix_normalized.loc[i, j] = new_weight

# Save
cd_matrix_normalized.to_csv('updated_cd_matrix_normalized.csv')

cd_matrix_normalized


In [ ]:
# Visualization
plt.figure(figsize=(15, 15))

# Calculate layout using spring_layout
pos = nx.spring_layout(G)

# Scale positions to increase spacing
pos = {node: (x * 2, y * 2) for (node, (x, y)) in pos.items()}

nx.draw_networkx_nodes(G, pos, node_size=700)
edges = G.edges(data=True)
nx.draw_networkx_edges(G, pos, edgelist=edges, width=[d['weight']*5 for u, v, d in edges])

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=12, font_family="sans-serif")

plt.axis("off")

# Save
save_path = 'Layout'
filename = 'Weighted_Network_Graph_2.png'
if not os.path.exists(save_path):
    os.makedirs(save_path)
plt.savefig(os.path.join(save_path, filename))
plt.show()


In [ ]:
os.makedirs('data', exist_ok=True)
filename = 'data/SWM_SC.csv'

connections = []

# Iterate over each row
for i in cd_matrix_normalized.index:
    connected_cds = cd_matrix_normalized.loc[i][cd_matrix_normalized.loc[i] > 0]
    for j, weight in connected_cds.items():
        connections.append([i, j, weight])

connections_df = pd.DataFrame(connections, columns=['Source_CD', 'Target_CD', 'Weight'])

# Save
connections_df.to_csv(filename, index=False)

file_saved_message = f"Spatial weights matrix saved as CSV at {filename}"
file_saved_message


In [ ]:
connections_df = pd.read_csv('data/SWM_SC.csv')

# Pivot the DataFrame to create a full matrix format
full_matrix_df = connections_df.pivot(index='Source_CD', columns='Target_CD', values='Weight')
full_matrix_df = full_matrix_df.fillna(0)
all_cds = sorted(set(connections_df['Source_CD']).union(connections_df['Target_CD']))
full_matrix_df = full_matrix_df.reindex(index=all_cds, columns=all_cds, fill_value=0)

# Save
full_matrix_filename = 'data/SWM_SC.csv'
full_matrix_df.to_csv(full_matrix_filename)

file_saved_message = f"Spatial weights matrix saved as CSV at {filename}"
file_saved_message


# Spatial Econometrics Model

## Step 1: Load Case Data and SWM

In [ ]:
# Load the spatial weight matrices
swm_sc_df = pd.read_csv('data/SWM_SC.csv', index_col=0)
swm_distance_df = pd.read_csv('data/SMW_Distance.csv', index_col=0)

# Align the rows and columns of both matrices
aligned_swm_sc_df = swm_sc_df.reindex(index=swm_distance_df.index, columns=swm_distance_df.columns)
aligned_swm_sc_df = aligned_swm_sc_df.fillna(0)
swm_combined_df = aligned_swm_sc_df.add(swm_distance_df, fill_value=0)

# Normalize
max_value = swm_combined_df.to_numpy().max()
swm_normalized_df = swm_combined_df / max_value

swm_normalized_filename = 'data/SWM.csv'
swm_normalized_df.to_csv(swm_normalized_filename)

print(f"The new spatial weights matrix 'SWM' has been saved to {swm_normalized_filename}")
swm_normalized_df


In [ ]:
# Load and convert the spatial weights matrix
def load_and_convert_swm(swm_filename):
    swm_df = pd.read_csv(swm_filename, index_col=0)
    swm_matrix = swm_df.to_numpy()
    neighbors = {i: swm_matrix[i].nonzero()[0].tolist() for i in range(len(swm_matrix))}
    weights = {i: swm_matrix[i, neighbors[i]].tolist() for i in range(len(swm_matrix))}
    return lps.weights.W(neighbors, weights)

swm = 'data/SWM.csv'
w = load_and_convert_swm(swm)
swm_distance = 'data/SMW_Distance.csv'
w_d = load_and_convert_swm(swm_distance)


## Step 2: Filter Variables

In [ ]:
CD_econ = gpd.read_file('data/ModelData/gdf_CD_econ.shp')

CD_econ = CD_econ.set_index('CD').loc[swm_normalized_df.columns.astype(str)].reset_index()
CD_econ.rename(columns={'index': 'CD'}, inplace=True)
CD_econ_filtered = CD_econ[CD_econ['CD'] != '414']

economic_indicators = [col for col in CD_econ.columns if col not in ['CD', 'geometry', 'Complaint', 'Summons', 'Shooting']]
dependent_vars = ['Complaint', 'Summons', 'Shooting']
aic_values = {}

print("GeoDataFrame:")
print(CD_econ.head())


### Reducing the dimensionality of the independent variable

#### Correlation-Based Feature Selection

In [ ]:
# Function for feature selection
def feature_selection(gdf, dependent_var):
    correlation_matrix = gdf.corr()
    correlation_with_dependent_var = correlation_matrix[dependent_var].sort_values(ascending=False)
    selected_features = correlation_with_dependent_var[np.abs(correlation_with_dependent_var) > 0.3].index.tolist()
    potential_dependent_vars = ['Complaint', 'Summons', 'Shooting']
    selected_features = [feature for feature in selected_features if feature not in potential_dependent_vars]
    return selected_features


#### Principal Component Analysis (PCA)

In [ ]:
# Apply PCA
def perform_pca(gdf, selected_features):
    X_selected = gdf[selected_features].values
    pca = PCA(n_components=5)
    X_pca = pca.fit_transform(X_selected)
    return X_pca, pca


#### Reducing the dimensionality

In [ ]:
dependent_vars = ['Complaint', 'Summons', 'Shooting']
# Apply feature selection for each dependent variable
selected_features_complaint = feature_selection(CD_econ_filtered, 'Complaint')
selected_features_summons = feature_selection(CD_econ_filtered, 'Summons')
selected_features_shooting = feature_selection(CD_econ_filtered, 'Shooting')
X_pca_complaint, pca_complaint = perform_pca(CD_econ_filtered, selected_features_complaint)
X_pca_summons, pca_summons = perform_pca(CD_econ_filtered, selected_features_summons)
X_pca_shooting, pca_shooting = perform_pca(CD_econ_filtered, selected_features_shooting)


In [ ]:
X_pca_complaint, pca_complaint, selected_features_complaint


In [ ]:
X_pca_summons, pca_summons, selected_features_summons


In [ ]:
X_pca_shooting, pca_shooting, selected_features_shooting


In [ ]:
# Function for triangular correlation matrix visualization without numbers
def plot_triangular_correlation_matrix(gdf, selected_features, title, filename):
    correlation_matrix = gdf[selected_features].corr()
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title(title)
    plt.savefig(f'Layout/{filename}.png')
    plt.show()

# Plot and save triangular correlation matrices
plot_triangular_correlation_matrix(CD_econ_filtered, selected_features_complaint, 'Correlation Matrix for Complaint', 'Complaint_Correlation_Matrix')
plot_triangular_correlation_matrix(CD_econ_filtered, selected_features_summons, 'Correlation Matrix for Summons', 'Summons_Correlation_Matrix')
plot_triangular_correlation_matrix(CD_econ_filtered, selected_features_shooting, 'Correlation Matrix for Shooting', 'Shooting_Correlation_Matrix')


## Step 3: Spatial Autocorrelation: Moran's I

In [ ]:
variables = ['Complaint', 'Summons', 'Shooting']
significant_vars_w = []
significant_vars_w_d = []

# Analyze with weight 'w'
for var in variables:
    moran_w = esda.Moran(CD_econ[var], w)
    print(f"Moran's I for {var} with 'w': {moran_w.I}, p-value: {moran_w.p_sim}")
    if moran_w.p_sim < 0.05:
        significant_vars_w.append((var, moran_w.p_sim))

# Analyze with weight 'w_d'
for var in variables:
    moran_w_d = esda.Moran(CD_econ[var], w_d)
    print(f"Moran's I for {var} with 'w_d': {moran_w_d.I}, p-value: {moran_w_d.p_sim}")
    if moran_w_d.p_sim < 0.05:
        significant_vars_w_d.append((var, moran_w_d.p_sim))

print(f"Statistically significant variables with 'w': {significant_vars_w}")
print(f"Statistically significant variables with 'w_d': {significant_vars_w_d}")


## Step 4: Geographically Weighted Regression (GWR)

In [ ]:
def perform_gwr(gdf, X_pca, pca, selected_features, w, dependent_var):
    y = gdf[dependent_var].values.reshape(-1, 1)
    ml_lag_model = ML_Lag(y, X_pca, w=w)
    summary = ml_lag_model.summary

    independent_features = [feat for feat in selected_features if feat != dependent_var]

    p_values = [item[1] for item in ml_lag_model.z_stat]
    coefficients = [item[0] for item in ml_lag_model.z_stat]
    significant_pca_vars = [i for i, p in enumerate(p_values[1:]) if p < 0.05]

    significant_original_vars_with_coeff = []
    num_pca_components = pca.components_.shape[0]

    for i in significant_pca_vars:
        if i < num_pca_components: 
            loadings = pca.components_[i]
            for j, loading in enumerate(loadings):
                if j < len(independent_features): 
                    original_var = independent_features[j]
                    coeff = coefficients[i+1] * loading
                    significant_original_vars_with_coeff.append((original_var, coeff))

    # Aggregate and sort coefficients
    aggregated_coeffs = {}
    for var, coeff in significant_original_vars_with_coeff:
        if var in aggregated_coeffs:
            aggregated_coeffs[var] += coeff
        else:
            aggregated_coeffs[var] = coeff

    sorted_significant_vars_with_coeff = sorted(aggregated_coeffs.items(), key=lambda x: abs(x[1]), reverse=True)

    return summary, sorted_significant_vars_with_coeff


def perform_gwr_analysis(gdf, X_pca, pca, selected_features, w, dependent_var, variables_dict):
    summary, significant_original_vars_with_coeff = perform_gwr(gdf, X_pca, pca, selected_features, w, dependent_var)
    p_value = variables_dict.get(dependent_var, None)
    significance_label = "" if dependent_var in variables_dict else " (Not statistically significant)"
    return summary, significant_original_vars_with_coeff, p_value, significance_label


In [ ]:
# Map each dependent variable to its corresponding X_pca, pca, and selected_features
dependent_vars_map = {
    'complaint': (X_pca_complaint, pca_complaint, selected_features_complaint),
    'summons': (X_pca_summons, pca_summons, selected_features_summons),
    'shooting': (X_pca_shooting, pca_shooting, selected_features_shooting)
}

# Initialize the dictionaries
variables_w = {var: p_value for var, p_value in significant_vars_w}
variables_w_d = {var: p_value for var, p_value in significant_vars_w_d}
gwr_results = {}
all_vars_to_analyze = set(variables_w.keys()) | set(variables_w_d.keys())

for var in all_vars_to_analyze:
    var_lower = var.lower()
    if var_lower in dependent_vars_map:
        X_pca_var, pca_var, selected_features_var = dependent_vars_map[var_lower]
        gwr_results[(var, 'w')] = perform_gwr_analysis(CD_econ_filtered, X_pca_var, pca_var, selected_features_var, w, var, variables_w)
        gwr_results[(var, 'w_d')] = perform_gwr_analysis(CD_econ_filtered, X_pca_var, pca_var, selected_features_var, w_d, var, variables_w_d)
    else:
        print(f"Variable {var} not found in dependent_vars_map.")

for var in all_vars_to_analyze:
    for weight in ['w', 'w_d']:
        result_key = (var, weight)
        if result_key in gwr_results:
            summary, coeffs, p_value, significance_label = gwr_results[result_key]
            p_value_text = f"(p-value: {p_value})" if p_value is not None else ""
            print(f"GWR Results for {var} with weight '{weight}' {p_value_text} {significance_label}:")
            print(summary)
            print(f"Significant Original Variables with Coefficients: {coeffs}")
            print("\n" + "=" * 50 + "\n")
        else:
            print(f"No results for {var} with weight '{weight}'.")
